> 尝试让Copilot自己整理，结果调整了文件路径，尤其前几天的gatrulation找不到了（在Git中没有add、commit），很难搞，解决方案：
> 
> （1）https://www.cnblogs.com/bq-med/articles/19123013从VSCode 本地历史恢复，还是不能完整恢复
> 
> （2）仔细在文件夹中查找，26.02.xx.gastrulation.ipynb，这些文件暂时保留并用.gitignore忽略这些文件及，避免文件丢失，起码仔细找还能找到，迁移好之后及时删除。有效的代码及时上传，不要怕仓库变大。（使用Github Copilot自动）

## Notebooks索引生成

此脚本用于扫描当前目录下的所有 `.ipynb` 文件，根据文件名 `时间-类别-内容.ipynb` 的格式进行分类，并生成可跳转的 Markdown 表格，最后将这些表格更新到 `README.md` 文件中。

In [5]:
import os
import re
from datetime import datetime
from collections import defaultdict

def generate_notebook_index():
    notebook_dir = '.'
    readme_path = os.path.join(notebook_dir, 'README.md')
    
    # 1. 扫描并分类Notebooks
    notebooks_by_category = defaultdict(list)
    known_categories = ['benchmark', 'case', 'quickstart']
    
    for filename in os.listdir(notebook_dir):
        if not filename.endswith('.ipynb') or filename == '26.03.10-generate_notebook_index.ipynb':
            continue
            
        match = re.match(r'(\d{2}\.\d{2}\.\d{2})-(.*?)-(.*)\.ipynb', filename)
        description = filename.replace('.ipynb', '').replace('_', ' ')
        category = 'Unknown' # Default category
        
        if match:
            date_str, category_raw, desc_part = match.groups()
            # Check if the extracted category is one of the known categories
            for known_cat in known_categories:
                if known_cat in category_raw.lower():
                    category = known_cat.capitalize()
                    break
            description = desc_part.replace('_', ' ')
        else:
            # For filenames that don't match the date-category-desc format
            # still check if they contain a known category keyword
            for known_cat in known_categories:
                if known_cat in filename.lower():
                    category = known_cat.capitalize()
                    break

        filepath = os.path.join(notebook_dir, filename)
        mod_time = datetime.fromtimestamp(os.path.getmtime(filepath)).strftime('%Y-%m-%d %H:%M:%S')
        
        notebooks_by_category[category].append({
            'filename': filename,
            'description': description,
            'mod_time': mod_time
        })
        
    # 2. 生成Markdown表格
    markdown_content = "\n## Notebook Tables\n"
    markdown_content += "方便您快速查找和访问不同类别的notebook。\n"

    # Ensure Unknown category is last
    sorted_categories = sorted(notebooks_by_category.keys(), key=lambda x: (x == 'Unknown', x))

    for category in sorted_categories:
        notebooks = notebooks_by_category[category]
        markdown_content += f"\n### {category}\n"
        markdown_content += "| filename | description | mod_time |\n"
        markdown_content += "|:---|:---|:---|\n"
        for nb in sorted(notebooks, key=lambda x: x['filename']):
            markdown_content += f"| [{nb['filename']}]({nb['filename']}) | {nb['description']} | {nb['mod_time']} |\n"
            
    # 3. 更新README.md
    try:
        with open(readme_path, 'r', encoding='utf-8') as f:
            readme_content = f.read()
        
        # 检查是否已存在 'Notebook Tables' 章节
        if '## Notebook Tables' in readme_content:
            # 替换旧的表格
            new_content = re.sub(r'## Notebook Tables[\s\S]*', markdown_content, readme_content, 1)
        else:
            # 追加新内容
            new_content = readme_content.strip() + '\n\n' + markdown_content
            
        with open(readme_path, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print(f"'{readme_path}' 已成功更新。")
        
    except FileNotFoundError:
        with open(readme_path, 'w', encoding='utf-8') as f:
            f.write(markdown_content)
        print(f"'{readme_path}' 已创建并写入索引。")

generate_notebook_index()

'./README.md' 已成功更新。


## TODO: 已有的描述要保持不变